# Inference using LSDB-style API

In [1]:
import lsdb

In [2]:
from distributed import Client

N_WORKERS, THREADS_PER_WORKER = 1, 1
client = Client(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)

## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [3]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [4]:
import pandas as pd
from lsdb_ml.hyrax.config_utils import ModelConfig

# The fields returned by the model for prediction
prediction_meta = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)

# ModelConfig provides the most important config parameters
# If the user needs more control, then they would use the LSDB-then-Hyrax approach.
# "Make the easy things easy and the hard things possible"
my_model_config = ModelConfig(
    model="HyraxLoopback",
    model_weights_file="./loopback_weights/fake_weights.pth",
    output_fields=prediction_meta,
    hyrax_config_file=None,  # Opt. - load a previous config, or provide complex configs
    batch_size=64,  # Optional
    primary_field_id="ticid",  # Optional
    input_fields=["ra_obj", "dec_obj", "lightcurve_sapflux"],  # Optional
)


## Context manager approach

Here we don't necessarily need to create the client (and cluster?) before hand,
instead we rely on the `ml_client` context manager to handle client creation and
plugin registration automatically.

The only required parameter for the `ml_client` context manager is the hyrax
configuration in order to instantiate the model on the worker.

Optional kwargs are passed directly to the `Client` constructor inside the context
manager.

In [ ]:
from lsdb_ml.hyrax.client import ml_client
from lsdb_ml.hyrax.worker_plugin import predict

with ml_client(my_model_config):  # + optional kwargs passed to Client(...)
    tess_pred = predict(tess_filtered_nans)  # the ideal setup
    tess_pred.compute()

    # tess_pred = tess_filtered_nans.map_partitions(
    #     predict,
    #     primary_id_field="ticid",  # optional, defaults to "_healpix_29"
    #     input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"],  # optional, defaults to all fields
    #     output_fields=PRED_META,
    #     meta=met,
    # )
    # tess_pred.compute()

/home/drew/miniconda3/envs/lsdb-ml/lib/python3.13/site-packages/distributed/node.py:195: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39049 instead
  warnings.warn(


http://127.0.0.1:39049/status


ValueError: No worker found

## Explicit client registration approach
This approach doesn't rely on a context manager and thus requires that the
Hyrax plugin be explicitly registered with the client before performing inference.
i.e. the user is responsible for creating the client and registering the plugin explicitly.

It also means that subsequent computation steps would be possible since the client
would not be closed out automatically.

In [ ]:
from lsdb_ml.hyrax.worker_plugin import predict, register_hyrax_plugin

register_hyrax_plugin(client, my_model_config)

tess_pred = predict(tess_filtered_nans)
# tess_pred = tess_filtered_nans.map_partitions(
#     predict,
#     primary_id_field="ticid",
#     input_fields=["ra_obj", "dec_obj", "lightcurve_sap_flux"],
#     output_fields=PRED_META,
#     meta=met,
# )

# Potentially perform additional computation steps here...

tess_pred.compute()

In [ ]:
client.close()